1- Install dependencies and get all DANSAR updates

In [ ]:
%cd /home/jovyan/uavsar/DANSAR
!git pull
!pip install -e .
!pip install asf-search requests numpy rasterio
!apt-get install -y -qq gdal-bin

2- User inputs

In [ ]:
from pathlib import Path
PRODUCT_NAME = "bigsur_14072_25007_008_250305_L090_CX_01"
OUTPUT_DIR = Path("/home/jovyan/uavsar/big_sur")

3- Import packages. If you see an error with DANSAR module, you might need to delete the DANSAR folder and clone it again

In [ ]:
import sys
%cd /home/jovyan/uavsar/DANSAR
sys.path.insert(0, "/home/jovyan/uavsar/DANSAR/src")
from urllib.parse import urlparse, parse_qs
from dansar.uavsar.infer_uavsar_paths import infer_uavsar_paths
from dansar.uavsar.download_uavsar import download_uavsar
from dansar.uavsar.ann2envi_header import ann2envi_header
from dansar.uavsar.make_tiff import (
    make_tiff,
    make_inc_mask_tiffs,
)
from dansar.utils.make_asf_session import make_asf_session
import numpy as np
import subprocess
import rasterio
import xml.etree.ElementTree as ET

4- Obtain ASF Authentication keys using your Earthdata credentials

In [ ]:
session, aws_credentials = make_asf_session()

import os

os.environ["AWS_ACCESS_KEY_ID"] = aws_credentials["accessKeyId"]
os.environ["AWS_SECRET_ACCESS_KEY"] = aws_credentials["secretAccessKey"]
os.environ["AWS_SESSION_TOKEN"] = aws_credentials["sessionToken"]
os.environ["AWS_DEFAULT_REGION"] = "us-west-2"

os.system("aws sts get-caller-identity")

5- Download GRDs from Alaska Satellite Facility

In [ ]:
import subprocess

PRODUCT_NAME = PRODUCT_NAME.strip()

PRODUCT_S3_PREFIX = (
    "s3://asf-cumulus-prod-products/"
    f"UAVSAR/UAVSAR_POLSAR/{PRODUCT_NAME}"
)

subprocess.run(
    [
        "aws",
        "s3",
        "sync",
        PRODUCT_S3_PREFIX,
        str(OUTPUT_DIR),
        "--exclude", "*",
        "--include", "*.ann",
        "--include", "*HHHH*.grd",
        "--include", "*HVHV*.grd",
        "--include", "*VVVV*.grd",
        "--include", "*.inc",
        "--only-show-errors",
    ],
    check=True,
)

6- Make ENVI headers for all GRDs in a folder

In [ ]:
from pathlib import Path

import numpy as np

from dansar.uavsar.ann2envi_header import ann2envi_header
from dansar.uavsar.make_tiff import make_tiff


OUTPUT_DIR = Path(OUTPUT_DIR)
PRODUCT_NAME = str(PRODUCT_NAME).strip()

OVERWRITE = False
NODATA = np.nan
CLIP_MIN = 1e-4

PRODUCT_PREFIX = PRODUCT_NAME.replace("_CX_01", "")
ANN_PATH = OUTPUT_DIR / f"{PRODUCT_NAME}.ann"

if not ANN_PATH.exists():
    raise FileNotFoundError(
        f"Missing annotation: {ANN_PATH}"
    )


# Find exactly one GRD per polarization
grd_files = []

for polarization in ("HHHH", "HVHV", "VVVV"):
    matches = sorted(
        OUTPUT_DIR.glob(
            f"{PRODUCT_PREFIX}{polarization}_*.grd"
        )
    )

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {polarization} GRD, "
            f"found {len(matches)}: {matches}"
        )

    grd_files.append(matches[0])


# Create all GRD headers and file-specific JSONs
metadata_outputs = ann2envi_header(
    ann_path=ANN_PATH,
    group="grd_pwr",
    output_dir=OUTPUT_DIR,
    save_json=True,
    overwrite=OVERWRITE,
)

# Create INC header and file-specific JSON
inc_metadata_outputs = ann2envi_header(
    ann_path=ANN_PATH,
    group="inc",
    output_dir=OUTPUT_DIR,
    save_json=True,
    overwrite=OVERWRITE,
)

for item in metadata_outputs:
    print("HDR ready:", item["header_path"].name)

    if item["json_path"] is not None:
        print("JSON ready:", item["json_path"].name)

7-Make TIFFs for all files in a folder

In [ ]:
# Create one TIFF per GRD
for grd_path in grd_files:
    hdr_path = Path(str(grd_path) + ".hdr")
    json_path = Path(str(grd_path) + ".json")
    tif_path = OUTPUT_DIR / f"{grd_path.stem}_cog.tif"
    inc_path = OUTPUT_DIR / f"{grd_path.stem}_cog.tif"

    if not hdr_path.exists():
        raise FileNotFoundError(
            f"Missing header: {hdr_path}"
        )

    if not json_path.exists():
        raise FileNotFoundError(
            f"Missing metadata JSON: {json_path}"
        )

    if tif_path.exists() and not OVERWRITE:
        print("TIFF exists:", tif_path.name)
        continue

    print("Making TIFF:", tif_path.name)

    written_tif = make_tiff(
        raster_path=grd_path,
        header_path=hdr_path,
        output_dir=OUTPUT_DIR,
        nodata=NODATA,
        invalid_min=0.0,
        clip_min=CLIP_MIN,
        overwrite=True,
        suffix="_cog",
        extension=".tif",
    )
    print("TIFF ready:", written_tif.name)
    

INC_PATH = OUTPUT_DIR / f"{PRODUCT_NAME}.inc"
INC_HDR_PATH = Path(str(INC_PATH) + ".hdr")
inc_tiff, mask_tiff = make_inc_mask_tiffs(
    inc_path=INC_PATH,
    header_path=INC_HDR_PATH,
    output_dir=OUTPUT_DIR,
    overwrite=OVERWRITE,
)

8- Make VRTs for all files in the user-specified product

In [ ]:
OUTPUT_DIR = Path(OUTPUT_DIR)
PRODUCT_NAME = str(PRODUCT_NAME).strip()

OVERWRITE = False

PRODUCT_PREFIX = PRODUCT_NAME.replace("_CX_01", "")

hhhh_cogs = sorted(OUTPUT_DIR.glob(f"{PRODUCT_PREFIX}HHHH_*_cog.tif"))
hvhv_cogs = sorted(OUTPUT_DIR.glob(f"{PRODUCT_PREFIX}HVHV_*_cog.tif"))
vvvv_cogs = sorted(OUTPUT_DIR.glob(f"{PRODUCT_PREFIX}VVVV_*_cog.tif"))

if len(hhhh_cogs) != 1:
    raise FileNotFoundError(f"Expected exactly one HHHH COG, found {len(hhhh_cogs)}: {hhhh_cogs}")

if len(hvhv_cogs) != 1:
    raise FileNotFoundError(f"Expected exactly one HVHV COG, found {len(hvhv_cogs)}: {hvhv_cogs}")

if len(vvvv_cogs) != 1:
    raise FileNotFoundError(f"Expected exactly one VVVV COG, found {len(vvvv_cogs)}: {vvvv_cogs}")

hhhh_tif = hhhh_cogs[0]
hvhv_tif = hvhv_cogs[0]
vvvv_tif = vvvv_cogs[0]

vrt_path = OUTPUT_DIR / f"{PRODUCT_NAME}_grd_pwr_stack.vrt"

if vrt_path.exists() and OVERWRITE:
    vrt_path.unlink()

if vrt_path.exists() and not OVERWRITE:
    pass
else:
    subprocess.run(
        [
            "gdalbuildvrt",
            "-separate",
            str(vrt_path),
            str(hhhh_tif),
            str(hvhv_tif),
            str(vvvv_tif),
        ],
        check=True,
    )

vrt_path